# DM4CT extended diffusion baselines for sparse-view CT

This notebook preserves the supplied DiffPDHG experiment as an optional, pinned reference path and runs four pixel-diffusion baselines through one DM4CT/AAPM/ASTRA pipeline:

`dps`, `red_diff`, `dmplug`, and `dds`.

The default `RUN_MODE="validation"` downloads and verifies `ct_slice_009.tif`, then runs the fixed DDS configuration once with live progress. `smoke` remains available for a short four-method execution check, and `full` requires the fixed 100 normalized AAPM test TIFFs configured below.


## Experiment contract

**Default paper protocol**

| Item | Exact value |
|---|---|
| Dataset | 2016 AAPM Low Dose CT Grand Challenge; DM4CT preprocessing |
| Full test slices | 100 fixed axial slices: `L506_000` through `L506_099` |
| Smoke slice | `L506_000` from the pinned DM4CT repository |
| Validation slices | One fixed user-supplied TIFF: `ct_slice_009` |
| Validation data source | `Seif-Hussein/daps_test` commit `d033e808902625c06a5d42037ddddd8da748e489`, downloaded and SHA-256 verified in validation mode |
| Image dimensions | `1 x 512 x 512` |
| Image normalization | Smoke/full: existing AAPM normalized floats in `[-1, 1]`. Validation: fixed reversible decode `2 * uint16 / 65535 - 1`; no per-image fitting. Metrics clip to `[-1, 1]`, `data_range=2`, no FOV mask |
| Diffusion checkpoint | `jiayangshi/lodochallenge_pixel_diffusion` at revision `b7e291c2febb28016a1ba1d639704822eb0d0793` |
| Projection angles | 80 values from DM4CT's `np.linspace(0, pi, 80)` convention |
| Detector bins | 512 |
| CT geometry | ASTRA `parallel3d`, one detector row, unit detector spacing, one 512 x 512 slice |
| Photon model | `I0=5000`, target average transmittance `0.5` |
| Cached observations | raw counts, normalized transmission, and DM4CT negative-log line integrals rescaled by the per-image attenuation factor |
| Random seed | `BASE_SEED=99`; measurement seed is `99 + image_index` |
| GPU | CUDA required; A100 preferred; the actual model is printed and saved at runtime |
| Pinned packages | diffusers 0.32.2, huggingface-hub 0.28.0, transformers 4.48.3, scikit-image 0.25.2, tifffile 2025.2.18, pandas 2.2.3, tqdm 4.67.1, LPIPS 0.1.4, ASTRA 2.5.0 |
| Repositories | DM4CT `49b3e5907178b56338d55c17944de17168a7a0a1`; supplied DiffPDHG branch `35db5260db51809e789e913ea481b069bada0ac9`; validation data `d033e808902625c06a5d42037ddddd8da748e489` |

PyTorch and its CUDA libraries are supplied by the Colab GPU runtime rather than reinstalled; their exact versions are asserted, printed, and stored in `environment_manifest.json`.

**Validation-data note.** The validation TIFF contains no physical-intensity or min/max metadata. The notebook therefore applies the fixed 16-bit full-scale decode to `ct_slice_009` without estimating scaling from its pixels. Validation runs DDS once, performs no hyperparameter sweep, does not execute DMPlug, and does not alter the frozen full-test configuration automatically.

**Difference from the supplied notebook.** The supplied notebook defaults to one of three small L067 demo TIFFs, a custom non-ASTRA differentiable projector, raw Poisson counts, `I0=10000`, and seed 99. It contains no DPS or RED-diff cells. This notebook therefore makes the requested paper protocol an explicit new default and keeps the original DiffPDHG launcher optional and separate instead of silently changing its historical results.


## 2. Runtime and GPU check

Select a GPU runtime in Colab. A CUDA GPU is a hard requirement because both ASTRA's direct 3D projector and the 512 x 512 diffusion model run on CUDA.


In [ ]:
import platform
import subprocess
import sys

import torch

print(subprocess.run(["nvidia-smi"], check=True, text=True, capture_output=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Colab select Runtime > Change runtime type > GPU.")
if tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError(f"PyTorch >=2.5 is required; found {torch.__version__}")
print(
    {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    }
)


## 3. Repository checkout and pinned environment installation

DM4CT is checked out at a commit that contains all four requested pipelines, including DDS. The supplied DiffPDHG repository is also pinned so its optional reference launcher and commit remain reproducible. ASTRA 2.5.0 is the sole intentional update from DM4CT's 2.3.0 environment because 2.5.0 provides current Colab/PyPI CUDA wheels while retaining the direct projector API used here.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DM4CT_URL = "https://github.com/DM4CT/DM4CT.git"
DM4CT_COMMIT = "49b3e5907178b56338d55c17944de17168a7a0a1"
DIFFPDHG_URL = "https://github.com/Seif-Hussein/dyscode.git"
DIFFPDHG_COMMIT = "35db5260db51809e789e913ea481b069bada0ac9"
CHECKPOINT_REVISION = "b7e291c2febb28016a1ba1d639704822eb0d0793"

DM4CT_DIR = Path("/content/DM4CT")
DIFFPDHG_DIR = Path("/content/dyscode_reference")

def checkout_pinned(url, directory, commit):
    directory = Path(directory)
    if not directory.exists():
        subprocess.run(
            ["git", "clone", "--filter=blob:none", "--no-checkout", url, str(directory)],
            check=True,
        )
    if not (directory / ".git").exists():
        raise RuntimeError(f"Existing path is not a git checkout: {directory}")
    subprocess.run(["git", "-C", str(directory), "fetch", "--depth", "1", "origin", commit], check=True)
    subprocess.run(["git", "-C", str(directory), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(
        ["git", "-C", str(directory), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != commit:
        raise RuntimeError(f"Commit mismatch for {directory}: {actual} != {commit}")
    return actual

checkout_pinned(DM4CT_URL, DM4CT_DIR, DM4CT_COMMIT)
checkout_pinned(DIFFPDHG_URL, DIFFPDHG_DIR, DIFFPDHG_COMMIT)

PIP_PACKAGES = [
    "astra-toolbox==2.5.0",
    "diffusers==0.32.2",
    "huggingface-hub==0.28.0",
    "transformers==4.48.3",
    "scikit-image==0.25.2",
    "tifffile==2025.2.18",
    "pandas==2.2.3",
    "tqdm==4.67.1",
    "lpips==0.1.4",
    "accelerate==1.2.1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PIP_PACKAGES], check=True)
print("Pinned repositories and Python packages are ready.")


## 4. Imports and reproducibility

Every measurement and method resets Python, NumPy, CPU PyTorch, and CUDA PyTorch seeds. cuDNN deterministic mode is enabled. ASTRA CUDA forward/backprojection can still use GPU accumulation orders that are not guaranteed bitwise deterministic across GPU models or driver versions; the manifest records both.


In [ ]:
import contextlib
import csv
import gc
import hashlib
import importlib.metadata
import json
import logging
import math
import os
import platform
import random
import shutil
import sys
import time
import traceback
import urllib.request
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import astra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from diffusers import DDIMScheduler, DDPMPipeline, DDPMScheduler
from diffusers.pipelines.pipeline_utils import ImagePipelineOutput
from diffusers.utils.torch_utils import randn_tensor
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tifffile import imread, imwrite
from tqdm.auto import tqdm

sys.path.insert(0, str(DM4CT_DIR))
from condition_methods import DDS, DMPlug, PosteriorSampling, RedDiff
from ct_reconstruction import fbp, sirt
from forward_operators_ct import NoNoise, Operator
from pipelines import (
    DDPMPipelineDDS,
    DDPMPipelineDMPlug,
    DDPMPipelineDPS,
    DDPMPipelineRedDiff,
)

DEVICE = torch.device("cuda")

def set_all_seeds(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = False

set_all_seeds(99)


## 5. Central configuration

Edit this cell only. Full-mode and validation-mode hyperparameters are frozen DM4CT reference settings. Validation performs no parameter sweep: it runs DDS once on `ct_slice_009` and records the resulting final-iterate metrics and compute.


In [ ]:
RUN_MODE = "validation"  # "smoke" | "validation" | "full"
METHODS = [
    "dps",
    "red_diff",
    "dmplug",
    "dds",
]

DATA_ROOT = "/content/drive/MyDrive/dm4ct_data/lodochallenge/test"
VALIDATION_ROOT = "/content/dm4ct_validation_data"
VALIDATION_DATA_REPOSITORY = "Seif-Hussein/daps_test"
VALIDATION_DATA_COMMIT = "d033e808902625c06a5d42037ddddd8da748e489"
VALIDATION_DATA_SUBDIR = "validation_data"
VALIDATION_FILE_SHA256 = {
    "ct_slice_009.tif": "82a1e8f431719dfbab0fb875a5d7e7a7e8a4ef8f8296c6963844528f43908031",
    "ct_slice_080.tif": "cc0ed562acbc604d62ac6c34c424b518e876db6ab0978f851decbe4e6cfe637d",
    "ct_slice_528.tif": "f6b4de802d3db582ab351caae3a00d263aa6c77cd8ecaa54e8783695bbf3bd05",
}
OUTPUT_ROOT = "/content/dm4ct_extended_outputs"
MODEL_PATH_OR_HF_ID = "jiayangshi/lodochallenge_pixel_diffusion"
NUM_ANGLES = 80
NUM_DETECTORS = 512
I0 = 5000.0
TRANSMITTANCE = 0.5
BASE_SEED = 99
SAVE_RECONSTRUCTIONS = True
RESUME = True
COMPUTE_LPIPS = False
ENABLE_POISSON_DMPLUG = False
ENABLE_LINEARIZED_TRACK = True
IMPLEMENTATION_REVISION = "dm4ct-extended-2026-07-24-v6-dds-source-provenance"

SMOKE_IMAGE_IDS = ["L506_000"]
VALIDATION_IMAGE_IDS = ["ct_slice_009"]
FULL_IMAGE_IDS = [f"L506_{index:03d}" for index in range(100)]
IMAGE_IDS = {
    "smoke": SMOKE_IMAGE_IDS,
    "validation": VALIDATION_IMAGE_IDS,
    "full": FULL_IMAGE_IDS,
}[RUN_MODE]
NUM_IMAGES = len(IMAGE_IDS)

MOUNT_GOOGLE_DRIVE = False
RUN_REFERENCE_DIFFPDHG = False
DIFFPDHG_METRICS_CSV = ""
DIFFPDHG_RECON_ROOT = ""
QUALITATIVE_IMAGE_IDS = ["L506_000", "L506_025", "L506_050", "L506_075"]

METHOD_CONFIGS = {
    "smoke": {
        "dps": {
            "num_inference_steps": 2,
            "scale": 10.0,
            "sirt_iterations": 2,
        },
        "red_diff": {
            "num_inference_steps": 2,
            "sigma": 1e-4,
            "loss_measurement_weight": 0.5,
            "loss_noise_weight": 10000.0,
            "learning_rate": 0.099,
            "sirt_iterations": 2,
        },
        "dmplug": {
            "num_inference_steps": 1,
            "optimization_iterations": 1,
            "epochs": 1,
            "learning_rate": 0.005,
            "initialization_seed": "BASE_SEED + image_index",
            "measurement_loss_type": "mse_line_integrals",
            "measurement_loss_weight": 1.0,
            "sirt_iterations": 2,
        },
        "dds": {
            "num_inference_steps": 2,
            "cg_inner": 1,
            "cg_eps": 1e-5,
            "gamma": 1.0,
            "eta": 0.85,
            "scheduler": "DDIMScheduler.from_config(checkpoint)",
        },
    },
    "validation": {
        "dps": {"num_inference_steps": 1000, "scale": 10.0, "sirt_iterations": 100},
        "red_diff": {
            "num_inference_steps": 200,
            "sigma": 1e-4,
            "loss_measurement_weight": 0.5,
            "loss_noise_weight": 10000.0,
            "learning_rate": 0.099,
            "sirt_iterations": 100,
        },
        "dmplug": {
            "num_inference_steps": 3,
            "optimization_iterations": 1000,
            "epochs": 1000,
            "learning_rate": 0.005,
            "initialization_seed": "BASE_SEED + image_index",
            "measurement_loss_type": "mse_line_integrals",
            "measurement_loss_weight": 1.0,
            "sirt_iterations": 100,
        },
        "dds": {
            "num_inference_steps": 100,
            "cg_inner": 5,
            "cg_eps": 1e-5,
            "gamma": 1.0,
            "eta": 0.85,
            "scheduler": "DDIMScheduler.from_config(checkpoint)",
        },
    },
    "full": {
        "dps": {"num_inference_steps": 1000, "scale": 10.0, "sirt_iterations": 100},
        "red_diff": {
            "num_inference_steps": 200,
            "sigma": 1e-4,
            "loss_measurement_weight": 0.5,
            "loss_noise_weight": 10000.0,
            "learning_rate": 0.099,
            "sirt_iterations": 100,
        },
        "dmplug": {
            "num_inference_steps": 3,
            "optimization_iterations": 1000,
            "epochs": 1000,
            "learning_rate": 0.005,
            "initialization_seed": "BASE_SEED + image_index",
            "measurement_loss_type": "mse_line_integrals",
            "measurement_loss_weight": 1.0,
            "sirt_iterations": 100,
        },
        "dds": {
            "num_inference_steps": 100,
            "cg_inner": 5,
            "cg_eps": 1e-5,
            "gamma": 1.0,
            "eta": 0.85,
            "scheduler": "DDIMScheduler.from_config(checkpoint)",
        },
    },
}

@dataclass(frozen=True)
class ExperimentConfig:
    RUN_MODE: str
    METHODS: List[str]
    DATA_ROOT: str
    VALIDATION_ROOT: str
    VALIDATION_DATA_REPOSITORY: str
    VALIDATION_DATA_COMMIT: str
    VALIDATION_DATA_SUBDIR: str
    VALIDATION_FILE_SHA256: Dict[str, str]
    OUTPUT_ROOT: str
    MODEL_PATH_OR_HF_ID: str
    NUM_IMAGES: int
    IMAGE_IDS: List[str]
    NUM_ANGLES: int
    NUM_DETECTORS: int
    I0: float
    TRANSMITTANCE: float
    BASE_SEED: int
    SAVE_RECONSTRUCTIONS: bool
    RESUME: bool
    COMPUTE_LPIPS: bool
    ENABLE_POISSON_DMPLUG: bool
    ENABLE_LINEARIZED_TRACK: bool
    IMPLEMENTATION_REVISION: str
    METHOD_CONFIGS: Dict[str, Dict[str, Any]]
    CHECKPOINT_REVISION: str
    DM4CT_COMMIT: str
    DIFFPDHG_COMMIT: str

CONFIG = ExperimentConfig(
    RUN_MODE=RUN_MODE,
    METHODS=METHODS,
    DATA_ROOT=DATA_ROOT,
    VALIDATION_ROOT=VALIDATION_ROOT,
    VALIDATION_DATA_REPOSITORY=VALIDATION_DATA_REPOSITORY,
    VALIDATION_DATA_COMMIT=VALIDATION_DATA_COMMIT,
    VALIDATION_DATA_SUBDIR=VALIDATION_DATA_SUBDIR,
    VALIDATION_FILE_SHA256=VALIDATION_FILE_SHA256,
    OUTPUT_ROOT=OUTPUT_ROOT,
    MODEL_PATH_OR_HF_ID=MODEL_PATH_OR_HF_ID,
    NUM_IMAGES=NUM_IMAGES,
    IMAGE_IDS=IMAGE_IDS,
    NUM_ANGLES=NUM_ANGLES,
    NUM_DETECTORS=NUM_DETECTORS,
    I0=I0,
    TRANSMITTANCE=TRANSMITTANCE,
    BASE_SEED=BASE_SEED,
    SAVE_RECONSTRUCTIONS=SAVE_RECONSTRUCTIONS,
    RESUME=RESUME,
    COMPUTE_LPIPS=COMPUTE_LPIPS,
    ENABLE_POISSON_DMPLUG=ENABLE_POISSON_DMPLUG,
    ENABLE_LINEARIZED_TRACK=ENABLE_LINEARIZED_TRACK,
    IMPLEMENTATION_REVISION=IMPLEMENTATION_REVISION,
    METHOD_CONFIGS=METHOD_CONFIGS[RUN_MODE],
    CHECKPOINT_REVISION=CHECKPOINT_REVISION,
    DM4CT_COMMIT=DM4CT_COMMIT,
    DIFFPDHG_COMMIT=DIFFPDHG_COMMIT,
)

config_payload = asdict(CONFIG)
CONFIG_HASH = hashlib.sha256(
    json.dumps(config_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]
OUTPUT_PATH = Path(CONFIG.OUTPUT_ROOT)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

print(json.dumps({**config_payload, "CONFIG_HASH": CONFIG_HASH}, indent=2))


In [ ]:
def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)

def atomic_write_json(path, payload):
    atomic_write_text(path, json.dumps(payload, indent=2, sort_keys=True, default=str))

def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

LOG_PATH = OUTPUT_PATH / "run_log.txt"
LOGGER = logging.getLogger("dm4ct_extended")
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()
stream_handler = logging.StreamHandler(sys.stdout)
file_handler = logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8")
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
stream_handler.setFormatter(formatter)
file_handler.setFormatter(formatter)
LOGGER.addHandler(stream_handler)
LOGGER.addHandler(file_handler)

REPOSITORY_HASHES = {
    "DM4CT": subprocess.check_output(
        ["git", "-C", str(DM4CT_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "DiffPDHG_reference": subprocess.check_output(
        ["git", "-C", str(DIFFPDHG_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "AAPM_pixel_diffusion_checkpoint": CHECKPOINT_REVISION,
    "Validation_data": CONFIG.VALIDATION_DATA_COMMIT,
}

ENVIRONMENT_MANIFEST = {
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "gpu_model": torch.cuda.get_device_name(0),
    "astra_version": getattr(astra, "__version__", "unknown"),
    "diffusers_version": importlib.metadata.version("diffusers"),
    "huggingface_hub_version": importlib.metadata.version("huggingface-hub"),
    "transformers_version": importlib.metadata.version("transformers"),
    "scikit_image_version": importlib.metadata.version("scikit-image"),
    "tqdm_version": importlib.metadata.version("tqdm"),
    "repository_commit_hashes": REPOSITORY_HASHES,
    "diffusion_checkpoint_identifier": CONFIG.MODEL_PATH_OR_HF_ID,
    "diffusion_checkpoint_revision": CHECKPOINT_REVISION,
    "determinism_note": (
        "Python/NumPy/PyTorch seeds and deterministic cuDNN are set. "
        "ASTRA/CUDA reductions may not be bitwise deterministic across hardware or drivers."
    ),
}
atomic_write_json(OUTPUT_PATH / "experiment_config.json", {**config_payload, "CONFIG_HASH": CONFIG_HASH})
atomic_write_json(OUTPUT_PATH / "environment_manifest.json", ENVIRONMENT_MANIFEST)
print(json.dumps(ENVIRONMENT_MANIFEST, indent=2))


## 6. Dataset and checkpoint loading

Smoke mode uses DM4CT's repository sample. Validation mode downloads `ct_slice_009.tif` from the immutable data commit when needed, verifies its SHA-256 digest, and decodes unsigned 16-bit full scale to `[-1, 1]` without fitting image-specific extrema. Full mode still fails immediately unless every explicitly listed normalized test TIFF is present. The single checkpoint is loaded once, and the same UNet object is passed to every pipeline.


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def prepare_validation_data(config):
    root = Path(config.VALIDATION_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    for image_id in config.IMAGE_IDS:
        filename = f"{image_id}.tif"
        expected_sha256 = config.VALIDATION_FILE_SHA256.get(filename)
        if expected_sha256 is None:
            raise KeyError(f"No pinned SHA-256 for validation file {filename}")
        target = root / filename
        if target.exists() and file_sha256(target) == expected_sha256:
            continue
        temporary = target.with_suffix(target.suffix + ".download")
        temporary.unlink(missing_ok=True)
        url = (
            f"https://raw.githubusercontent.com/{config.VALIDATION_DATA_REPOSITORY}/"
            f"{config.VALIDATION_DATA_COMMIT}/{config.VALIDATION_DATA_SUBDIR}/{filename}"
        )
        try:
            with urllib.request.urlopen(url) as response, temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
            actual_sha256 = file_sha256(temporary)
            if actual_sha256 != expected_sha256:
                raise RuntimeError(
                    f"Validation download checksum mismatch for {filename}: "
                    f"{actual_sha256} != {expected_sha256}"
                )
            os.replace(temporary, target)
        except Exception:
            temporary.unlink(missing_ok=True)
            raise
    return root

def resolve_image_records(config):
    if config.RUN_MODE == "smoke":
        root = DM4CT_DIR / "lodochallenge"
    elif config.RUN_MODE == "validation":
        root = prepare_validation_data(config)
    else:
        root = Path(config.DATA_ROOT)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist for {config.RUN_MODE}: {root}")

    records = []
    for image_id in config.IMAGE_IDS:
        candidates = [root / f"{image_id}.tif", root / f"{image_id}.tiff"]
        matches = [path for path in candidates if path.exists()]
        if len(matches) != 1:
            raise FileNotFoundError(
                f"Expected exactly one TIFF for {image_id} under {root}; found {matches}"
            )
        records.append({"image_id": image_id, "path": matches[0]})
    if len(records) != config.NUM_IMAGES:
        raise RuntimeError("Resolved image count does not match NUM_IMAGES")
    return records

def load_normalized_ground_truth(path):
    source = np.asarray(imread(path))
    source = np.squeeze(source)
    if source.shape != (512, 512):
        raise ValueError(f"Inconsistent image dimensions for {path}: {source.shape}, expected (512, 512)")
    if source.dtype == np.uint16:
        array = source.astype(np.float32) * (2.0 / 65535.0) - 1.0
    elif np.issubdtype(source.dtype, np.floating):
        array = source.astype(np.float32)
    else:
        raise TypeError(
            f"Unsupported TIFF dtype for {path}: {source.dtype}; expected uint16 or normalized float"
        )
    if not np.isfinite(array).all():
        raise ValueError(f"Nonfinite ground truth: {path}")
    if float(array.min()) < -1.001 or float(array.max()) > 1.001:
        raise ValueError(
            f"Model-normalization mismatch for {path}: range [{array.min()}, {array.max()}], expected [-1, 1]"
        )
    return torch.from_numpy(array).unsqueeze(0).contiguous()

IMAGE_RECORDS = resolve_image_records(CONFIG)

model_source = CONFIG.MODEL_PATH_OR_HF_ID
model_kwargs = {"torch_dtype": torch.float32}
if Path(model_source).is_absolute():
    if not Path(model_source).exists():
        raise FileNotFoundError(f"Missing local checkpoint: {model_source}")
    model_kwargs["local_files_only"] = True
else:
    model_kwargs["revision"] = CHECKPOINT_REVISION

BASE_PIPELINE = DDPMPipeline.from_pretrained(model_source, **model_kwargs)
DIFFUSION_UNET = BASE_PIPELINE.unet.to(DEVICE).eval()
DIFFUSION_UNET.requires_grad_(False)
BASE_SCHEDULER_CONFIG = dict(BASE_PIPELINE.scheduler.config)
if int(DIFFUSION_UNET.config.sample_size) != 512 or int(DIFFUSION_UNET.config.in_channels) != 1:
    raise RuntimeError(
        f"Checkpoint mismatch: sample_size={DIFFUSION_UNET.config.sample_size}, "
        f"in_channels={DIFFUSION_UNET.config.in_channels}"
    )
del BASE_PIPELINE
gc.collect()
torch.cuda.empty_cache()
print(f"Loaded one shared pixel-space checkpoint on {torch.cuda.get_device_name(0)}")


## 7. ASTRA operator construction

The geometry follows DM4CT's actual `parallel3d` construction. Shape checks are fatal. Operator initialization calls used to form SIRT normalization arrays occur before any method timer and are not charged to a reconstruction.


In [ ]:
ANGLES = np.linspace(0.0, np.pi, CONFIG.NUM_ANGLES, dtype=np.float32)
VOLUME_GEOMETRY = astra.create_vol_geom(512, 512, 1)
PROJECTION_GEOMETRY = astra.create_proj_geom(
    "parallel3d", 1.0, 1.0, 1, CONFIG.NUM_DETECTORS, ANGLES
)
BASE_OPERATOR = Operator(
    volume_geometry=VOLUME_GEOMETRY,
    projection_geometry=PROJECTION_GEOMETRY,
)
expected_volume_shape = (1, 512, 512)
expected_projection_shape = (1, CONFIG.NUM_ANGLES, CONFIG.NUM_DETECTORS)
if tuple(BASE_OPERATOR.volume_shape) != expected_volume_shape:
    raise RuntimeError(
        f"ASTRA volume shape mismatch: {BASE_OPERATOR.volume_shape} != {expected_volume_shape}"
    )
if tuple(BASE_OPERATOR.projection_shape) != expected_projection_shape:
    raise RuntimeError(
        f"ASTRA projection shape mismatch: {BASE_OPERATOR.projection_shape} != {expected_projection_shape}"
    )
print(
    {
        "geometry": "parallel3d",
        "angles": CONFIG.NUM_ANGLES,
        "detector_bins": CONFIG.NUM_DETECTORS,
        "volume_shape": BASE_OPERATOR.volume_shape,
        "projection_shape": BASE_OPERATOR.projection_shape,
    }
)


## 8. Measurement-model audit

Inspection of `forward_operators_ct.PoissonNoise` at DM4CT commit `49b3e59...` shows that its variable `measurement` is **not raw photon counts**. DM4CT projects a normalized image to line integrals, rescales them to target transmittance, samples photons, applies `-log(counts/I0)`, and divides by the same scale factor. The returned tensor is therefore a noisy, negative-log, rescaled line-integral sinogram.

| Method | Tensor received | Forward/data-consistency model | Raw nonlinear likelihood? | Strictly likelihood-matched to supplied DiffPDHG? |
|---|---|---|---|---|
| DPS | cached `log_line_integrals` | ASTRA linear `A(x)` with DM4CT normalized residual guidance | No | No |
| RED-diff | cached `log_line_integrals` | ASTRA linear `A(x)` with squared line-integral loss | No | No |
| `dmplug_linearized` | cached `log_line_integrals` | differentiable diffusion generator plus ASTRA linear MSE | No | No |
| `dmplug_poisson` (optional) | cached raw photon counts | `lambda(x)=I0*exp(-s*A(x))`, Poisson NLL through the same generator | Yes | Still no: the supplied reference uses a different non-ASTRA projector and attenuation mapping |
| `DDS (log-linearized)` | cached `log_line_integrals` | linear ASTRA normal equations with CG | No | No |
| Supplied DiffPDHG reference | raw photon counts | custom non-ASTRA projector, `mu(x)` mapping, Beer-Lambert Poisson objective | Yes | Reference likelihood itself |

The primary strict table contains the four DM4CT linearized paths. The optional Poisson DMPlug path is placed only in the extended table and carries an explicit measurement-model label. DDS is never presented as a raw-count method.


## 9. Shared measurement generation and caching

Each image is projected and noised once. The bundle is written atomically before any method runs and contains ground truth, noiseless line integrals, raw counts, normalized transmission, log-linearized observation, seed, attenuation scale, and geometry metadata. All methods receive tensors loaded from this same bundle.


In [ ]:
MEASUREMENT_DIR = OUTPUT_PATH / "measurements"
MEASUREMENT_DIR.mkdir(parents=True, exist_ok=True)

measurement_contract = {
    "num_angles": CONFIG.NUM_ANGLES,
    "num_detectors": CONFIG.NUM_DETECTORS,
    "I0": CONFIG.I0,
    "transmittance": CONFIG.TRANSMITTANCE,
    "geometry": "parallel3d",
    "dm4ct_commit": DM4CT_COMMIT,
    "normalization": (
        "uint16 full scale: 2*x/65535 - 1"
        if CONFIG.RUN_MODE == "validation"
        else "global AAPM min/max -> [-1, 1]"
    ),
    "validation_data_commit": (
        CONFIG.VALIDATION_DATA_COMMIT if CONFIG.RUN_MODE == "validation" else None
    ),
}
MEASUREMENT_HASH = hashlib.sha256(
    json.dumps(measurement_contract, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)

def load_torch_bundle(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def validate_measurement_bundle(bundle, image_id):
    required = [
        "ground_truth",
        "noiseless_projection",
        "raw_counts",
        "normalized_transmission",
        "log_line_integrals",
    ]
    if bundle.get("image_id") != image_id or bundle.get("measurement_hash") != MEASUREMENT_HASH:
        raise ValueError("Measurement cache hash or image ID mismatch")
    for key in required:
        tensor = bundle[key]
        if not torch.is_tensor(tensor) or not torch.isfinite(tensor).all():
            raise ValueError(f"Invalid cached tensor {key} for {image_id}")
    if tuple(bundle["ground_truth"].shape) != expected_volume_shape:
        raise ValueError(f"Cached ground-truth shape mismatch for {image_id}")
    if tuple(bundle["log_line_integrals"].shape) != expected_projection_shape:
        raise ValueError(f"Cached measurement shape mismatch for {image_id}")

def generate_or_load_measurement(record, image_index):
    image_id = record["image_id"]
    path = MEASUREMENT_DIR / f"{image_id}.pt"
    if CONFIG.RESUME and path.exists():
        try:
            bundle = load_torch_bundle(path)
            validate_measurement_bundle(bundle, image_id)
            LOGGER.info("Loaded shared measurement cache for %s", image_id)
            return bundle
        except Exception:
            LOGGER.exception("Invalid measurement cache; regenerating %s", image_id)

    measurement_seed = CONFIG.BASE_SEED + image_index
    set_all_seeds(measurement_seed)
    ground_truth = load_normalized_ground_truth(record["path"]).to(DEVICE)
    with torch.no_grad():
        noiseless = BASE_OPERATOR.forward(ground_truth).clone()
        mean_sinogram = noiseless.mean()
        if not torch.isfinite(mean_sinogram) or abs(float(mean_sinogram)) < 1e-12:
            raise RuntimeError(f"Cannot determine attenuation scale for {image_id}")
        attenuation_scale = -math.log(CONFIG.TRANSMITTANCE) / float(mean_sinogram)
        scaled_line_integrals = noiseless * attenuation_scale
        expected_counts = torch.exp(-scaled_line_integrals) * CONFIG.I0
        if not torch.isfinite(expected_counts).all() or (expected_counts < 0).any():
            raise RuntimeError(f"Nonfinite photon rates for {image_id}")
        generator = torch.Generator(device=DEVICE).manual_seed(measurement_seed)
        raw_counts = torch.poisson(expected_counts, generator=generator)
        counts_for_log = raw_counts.clone()
        counts_for_log[counts_for_log == 0] = 1.0
        normalized_transmission = counts_for_log / CONFIG.I0
        log_line_integrals = -torch.log(normalized_transmission) / attenuation_scale

    bundle = {
        "image_id": image_id,
        "source_path": str(record["path"]),
        "measurement_hash": MEASUREMENT_HASH,
        "seed": measurement_seed,
        "ground_truth": ground_truth.detach().cpu(),
        "noiseless_projection": noiseless.detach().cpu(),
        "raw_counts": raw_counts.detach().cpu(),
        "expected_counts": expected_counts.detach().cpu(),
        "normalized_transmission": normalized_transmission.detach().cpu(),
        "log_line_integrals": log_line_integrals.detach().cpu(),
        "attenuation_scale": float(attenuation_scale),
        "actual_average_transmission": float(normalized_transmission.mean()),
        "geometry_metadata": {
            **measurement_contract,
            "volume_shape": list(expected_volume_shape),
            "projection_shape": list(expected_projection_shape),
            "angles": ANGLES.tolist(),
        },
    }
    validate_measurement_bundle(bundle, image_id)
    atomic_torch_save(bundle, path)
    LOGGER.info("Generated and cached shared measurement for %s with seed %d", image_id, measurement_seed)
    return bundle

MEASUREMENT_BUNDLES = {
    record["image_id"]: generate_or_load_measurement(record, image_index)
    for image_index, record in enumerate(IMAGE_RECORDS)
}
print(
    [
        {
            "image_id": image_id,
            "seed": bundle["seed"],
            "actual_average_transmission": bundle["actual_average_transmission"],
            "cache": str(MEASUREMENT_DIR / f"{image_id}.pt"),
        }
        for image_id, bundle in MEASUREMENT_BUNDLES.items()
    ]
)


## 10. Unified metric utilities

Metrics are computed only from the returned final iterate. Ground truth and reconstruction are clipped identically to `[-1, 1]`; PSNR and SSIM use `data_range=2`, matching DM4CT's normalized-domain evaluation. No best-iteration or oracle stopping enters the primary CSVs.


In [ ]:
_LPIPS_MODEL = None

def final_metric_arrays(ground_truth, reconstruction):
    ground_truth = np.asarray(ground_truth, dtype=np.float32).squeeze()
    reconstruction = np.asarray(reconstruction, dtype=np.float32).squeeze()
    if ground_truth.shape != (512, 512) or reconstruction.shape != (512, 512):
        raise ValueError(f"Metric shape mismatch: gt={ground_truth.shape}, recon={reconstruction.shape}")
    if not np.isfinite(reconstruction).all():
        raise FloatingPointError("Nonfinite final reconstruction")
    return np.clip(ground_truth, -1.0, 1.0), np.clip(reconstruction, -1.0, 1.0)

def compute_final_metrics(ground_truth, reconstruction, compute_lpips=False):
    global _LPIPS_MODEL
    ground_truth, reconstruction = final_metric_arrays(ground_truth, reconstruction)
    values = {
        "psnr": float(peak_signal_noise_ratio(ground_truth, reconstruction, data_range=2.0)),
        "ssim": float(structural_similarity(ground_truth, reconstruction, data_range=2.0)),
        "lpips": np.nan,
    }
    if compute_lpips:
        import lpips
        if _LPIPS_MODEL is None:
            _LPIPS_MODEL = lpips.LPIPS(net="alex").to(DEVICE).eval()
        gt_tensor = torch.from_numpy(ground_truth)[None, None].to(DEVICE).repeat(1, 3, 1, 1)
        rec_tensor = torch.from_numpy(reconstruction)[None, None].to(DEVICE).repeat(1, 3, 1, 1)
        with torch.inference_mode():
            values["lpips"] = float(_LPIPS_MODEL(gt_tensor, rec_tensor).item())
    return values


## 11. NFE and operator-call counters

NFE is a forward-pre-hook count on the shared UNet, so repeated differentiable DMPlug trajectories are counted in full. No dummy or warm-up UNet evaluation is performed: timing and NFE begin with the first real reconstruction operation. The operator wrapper counts forward and adjoint calls, including SIRT initialization and method diagnostics. DDS reports CG iterations separately.


In [ ]:
class ModelForwardCounter:
    def __init__(self, module):
        self.count = 0
        self.handle = module.register_forward_pre_hook(self._hook)

    def _hook(self, module, inputs):
        self.count += 1

    def reset(self):
        self.count = 0

    def close(self):
        self.handle.remove()

class CountingOperator:
    def __init__(self, base, sirt_iterations=100):
        self.base = base
        self.sirt_iterations = int(sirt_iterations)
        self.forward_calls = 0
        self.adjoint_calls = 0

    def __getattr__(self, name):
        return getattr(self.base, name)

    def reset_counts(self):
        self.forward_calls = 0
        self.adjoint_calls = 0

    def __call__(self, volume):
        self.forward_calls += 1
        projection = self.base(volume)
        if projection.requires_grad:
            backward_calls = volume.shape[0] if volume.ndim == 4 else 1

            def count_autograd_adjoint(gradient):
                self.adjoint_calls += int(backward_calls)
                return gradient

            projection.register_hook(count_autograd_adjoint)
        return projection

    def forward(self, volume, **kwargs):
        return self(volume)

    def T(self, projection):
        self.adjoint_calls += 1
        return self.base.T(projection)

    def transpose(self, projection):
        return self.T(projection)

    def project(self, volume, projection):
        return (
            volume
            - self.base.C * self.transpose(self.base.R * self.forward(volume))
            + self.base.C * self.transpose(self.base.R * projection)
        )

    def pseudo_inverse(
        self,
        projection,
        method="sirt",
        num_iterations=None,
        min_constraint=None,
        max_constraint=None,
        x_init=None,
    ):
        iterations = self.sirt_iterations if num_iterations is None else int(num_iterations)
        if method == "fbp":
            return fbp(self, projection)
        if method != "sirt":
            raise NotImplementedError(method)
        if projection.ndim == 4:
            volumes = torch.zeros(
                (projection.shape[0], *self.volume_shape),
                dtype=torch.float32,
                device=projection.device,
            )
            for batch_index in range(projection.shape[0]):
                volumes[batch_index] = sirt(
                    self,
                    projection[batch_index],
                    iterations,
                    min_constraint,
                    max_constraint,
                    x_init,
                )
            return volumes
        if projection.ndim == 3:
            return sirt(
                self, projection, iterations, min_constraint, max_constraint, x_init
            )
        raise ValueError(f"Unsupported projection rank: {projection.ndim}")

NFE_COUNTER = ModelForwardCounter(DIFFUSION_UNET)

@dataclass
class MethodResult:
    reconstruction: Optional[np.ndarray]
    runtime_seconds: float
    nfe: int
    method_hyperparameters: Dict[str, Any]
    measurement_model: str
    num_forward_operator_calls: int
    num_adjoint_operator_calls: int
    convergence_diagnostics: Dict[str, Any]
    failure_status: str
    error_message: str


## 12. Existing DPS and RED-diff adapters

These adapters instantiate the pinned DM4CT pipeline and conditioning classes without changing their algorithms. The only smoke-mode reduction is the centrally declared number of diffusion/SIRT steps.


In [ ]:
def make_generator(seed):
    return torch.Generator(device=DEVICE).manual_seed(int(seed))

def bundle_tensor(bundle, key, add_batch=True):
    tensor = bundle[key].to(DEVICE, dtype=torch.float32)
    return tensor.unsqueeze(0) if add_batch else tensor

def output_to_tensor(output):
    array = np.asarray(output.images, dtype=np.float32)
    tensor = torch.from_numpy(array).to(DEVICE)
    if tuple(tensor.shape) != (1, 1, 512, 512):
        raise ValueError(f"Pipeline output shape mismatch: {tuple(tensor.shape)}")
    if not torch.isfinite(tensor).all():
        raise FloatingPointError("Pipeline returned a nonfinite reconstruction")
    return tensor

def relative_line_integral_residual(operator, reconstruction, measurement):
    predicted = operator.forward(reconstruction)
    return float(
        (torch.linalg.vector_norm(predicted - measurement) /
         torch.linalg.vector_norm(measurement).clamp_min(1e-12)).detach().cpu()
    )

def timed_work(_method_key, operator, work):
    NFE_COUNTER.reset()
    operator.reset_counts()
    torch.cuda.synchronize()
    started = time.perf_counter()
    payload = work()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    return payload, elapsed, NFE_COUNTER.count, operator.forward_calls, operator.adjoint_calls

def run_dps(ground_truth, measurement_bundle, operator_bundle, diffusion_model, method_config, seed):
    operator = CountingOperator(
        operator_bundle, sirt_iterations=method_config["sirt_iterations"]
    )
    scheduler = DDPMScheduler.from_config(BASE_SCHEDULER_CONFIG)
    pipeline = DDPMPipelineDPS(unet=diffusion_model, scheduler=scheduler).to(DEVICE)
    pipeline.set_progress_bar_config(disable=True)
    pipeline.measurement_condition = PosteriorSampling(
        operator=operator, noiser=NoNoise(), scale=method_config["scale"]
    )
    measurement = bundle_tensor(measurement_bundle, "log_line_integrals")

    def work():
        output = pipeline(
            num_inference_steps=method_config["num_inference_steps"],
            measurement=measurement,
            generator=make_generator(seed),
            output_type="np",
        )
        reconstruction = output_to_tensor(output)
        residual = relative_line_integral_residual(operator, reconstruction, measurement)
        return reconstruction[0, 0].detach().cpu().numpy(), residual

    payload, runtime, nfe, forward_calls, adjoint_calls = timed_work("dps", operator, work)
    del pipeline
    return MethodResult(
        reconstruction=payload[0],
        runtime_seconds=runtime,
        nfe=nfe,
        method_hyperparameters={**method_config, "initialization_seed": seed},
        measurement_model="log_linearized_poisson",
        num_forward_operator_calls=forward_calls,
        num_adjoint_operator_calls=adjoint_calls,
        convergence_diagnostics={"final_relative_line_integral_residual": payload[1]},
        failure_status="success",
        error_message="",
    )

def run_red_diff(ground_truth, measurement_bundle, operator_bundle, diffusion_model, method_config, seed):
    operator = CountingOperator(
        operator_bundle, sirt_iterations=method_config["sirt_iterations"]
    )
    scheduler = DDIMScheduler.from_config(BASE_SCHEDULER_CONFIG)
    pipeline = DDPMPipelineRedDiff(unet=diffusion_model, scheduler=scheduler).to(DEVICE)
    pipeline.set_progress_bar_config(disable=True)
    pipeline.measurement_condition = RedDiff(operator=operator, noiser=NoNoise())
    measurement = bundle_tensor(measurement_bundle, "log_line_integrals")

    def work():
        output = pipeline(
            num_inference_steps=method_config["num_inference_steps"],
            measurement=measurement,
            generator=make_generator(seed),
            sigma=method_config["sigma"],
            loss_measurement_weight=method_config["loss_measurement_weight"],
            loss_noise_weight=method_config["loss_noise_weight"],
            lr=method_config["learning_rate"],
            output_type="np",
        )
        reconstruction = output_to_tensor(output)
        residual = relative_line_integral_residual(operator, reconstruction, measurement)
        return reconstruction[0, 0].detach().cpu().numpy(), residual

    payload, runtime, nfe, forward_calls, adjoint_calls = timed_work(
        "red_diff", operator, work
    )
    del pipeline
    return MethodResult(
        reconstruction=payload[0],
        runtime_seconds=runtime,
        nfe=nfe,
        method_hyperparameters={**method_config, "initialization_seed": seed},
        measurement_model="log_linearized_poisson",
        num_forward_operator_calls=forward_calls,
        num_adjoint_operator_calls=adjoint_calls,
        convergence_diagnostics={"final_relative_line_integral_residual": payload[1]},
        failure_status="success",
        error_message="",
    )


## 13. DMPlug adapter

The primary path is the official `DDPMPipelineDMPlug` plus `DMPlug`. A recording criterion captures one optimization loss per epoch. When enabled, `dmplug_poisson` uses the same pipeline but replaces line-integral MSE with a raw-count Poisson NLL; it is stored under a distinct method and measurement-model label.


In [ ]:
class RecordingLoss(nn.Module):
    def __init__(self, base_loss, weight=1.0, progress_callback=None):
        super().__init__()
        self.base_loss = base_loss
        self.weight = float(weight)
        self.progress_callback = progress_callback
        self.values = []

    def forward(self, prediction, target):
        value = self.weight * self.base_loss(prediction, target)
        scalar_value = float(value.detach().cpu())
        self.values.append(scalar_value)
        if self.progress_callback is not None:
            self.progress_callback(scalar_value)
        return value

class PoissonCountNLL(nn.Module):
    def __init__(self, incident_intensity, attenuation_scale):
        super().__init__()
        self.incident_intensity = float(incident_intensity)
        self.attenuation_scale = float(attenuation_scale)

    def forward(self, predicted_line_integrals, raw_counts):
        predicted_counts = self.incident_intensity * torch.exp(
            -self.attenuation_scale * predicted_line_integrals
        )
        predicted_counts = predicted_counts.clamp_min(1e-8)
        return (predicted_counts - raw_counts * torch.log(predicted_counts)).mean()

def poisson_count_residual(operator, reconstruction, raw_counts, attenuation_scale):
    predicted_line_integrals = operator.forward(reconstruction)
    predicted_counts = CONFIG.I0 * torch.exp(-attenuation_scale * predicted_line_integrals)
    return float(
        (
            torch.sqrt(torch.mean((predicted_counts - raw_counts) ** 2))
            / raw_counts.mean().clamp_min(1.0)
        ).detach().cpu()
    )

def run_dmplug(
    ground_truth,
    measurement_bundle,
    operator_bundle,
    diffusion_model,
    method_config,
    seed,
    poisson=False,
):
    operator = CountingOperator(
        operator_bundle, sirt_iterations=method_config["sirt_iterations"]
    )
    scheduler = DDIMScheduler.from_config(BASE_SCHEDULER_CONFIG)
    pipeline = DDPMPipelineDMPlug(unet=diffusion_model, scheduler=scheduler).to(DEVICE)
    pipeline.set_progress_bar_config(disable=True)
    pipeline.measurement_condition = DMPlug(operator=operator, noiser=NoNoise())

    if poisson:
        measurement = bundle_tensor(measurement_bundle, "raw_counts")
        base_loss = PoissonCountNLL(
            CONFIG.I0, measurement_bundle["attenuation_scale"]
        )
        measurement_model = "raw_poisson_counts_beer_lambert"
        method_key = "dmplug_poisson"
        loss_type = "poisson_count_nll"
    else:
        measurement = bundle_tensor(measurement_bundle, "log_line_integrals")
        base_loss = nn.MSELoss()
        measurement_model = "log_linearized_poisson"
        method_key = "dmplug_linearized"
        loss_type = "mse_line_integrals"

    optimizer_progress = (
        tqdm(
            total=method_config["optimization_iterations"],
            desc=f"{method_key}: optimizer",
            unit="iteration",
            position=1,
            leave=False,
            dynamic_ncols=True,
        )
        if CONFIG.RUN_MODE == "validation"
        else None
    )

    def update_optimizer_progress(loss_value):
        if optimizer_progress is not None:
            optimizer_progress.set_postfix(loss=f"{loss_value:.4g}")
            optimizer_progress.update(1)

    criterion = RecordingLoss(
        base_loss,
        weight=method_config["measurement_loss_weight"],
        progress_callback=update_optimizer_progress,
    )

    def work():
        try:
            output = pipeline(
                num_inference_steps=method_config["num_inference_steps"],
                measurement=measurement,
                epochs=method_config["optimization_iterations"],
                lr=method_config["learning_rate"],
                criterion=criterion,
                generator=make_generator(seed),
                output_type="np",
            )
            reconstruction = output_to_tensor(output)
            if poisson:
                residual = poisson_count_residual(
                    operator,
                    reconstruction,
                    measurement,
                    measurement_bundle["attenuation_scale"],
                )
            else:
                residual = relative_line_integral_residual(
                    operator, reconstruction, measurement
                )
            return reconstruction[0, 0].detach().cpu().numpy(), residual
        finally:
            if optimizer_progress is not None:
                optimizer_progress.close()

    payload, runtime, nfe, forward_calls, adjoint_calls = timed_work(
        method_key, operator, work
    )
    del pipeline
    hyperparameters = {
        **method_config,
        "initialization_seed": seed,
        "measurement_loss_type": loss_type,
    }
    return MethodResult(
        reconstruction=payload[0],
        runtime_seconds=runtime,
        nfe=nfe,
        method_hyperparameters=hyperparameters,
        measurement_model=measurement_model,
        num_forward_operator_calls=forward_calls,
        num_adjoint_operator_calls=adjoint_calls,
        convergence_diagnostics={
            "optimization_loss_trajectory": criterion.values,
            "final_measurement_residual": payload[1],
            "completed_optimizer_iterations": len(criterion.values),
        },
        failure_status="success",
        error_message="",
    )


## 14. DDS adapter

`DiagnosticDDSPipeline` is a line-for-line instrumentation of the pinned DM4CT `DDPMPipelineDDS` loop. It keeps the official DDIM and CG updates but exposes residuals, completed CG iterations, and divergence state that the upstream class does not return.


In [ ]:
class DiagnosticDDSPipeline(DDPMPipelineDDS):
    def __call__(
        self,
        batch_size=1,
        generator=None,
        num_inference_steps=100,
        output_type="np",
        measurement=None,
        return_dict=True,
        eta=0.85,
        cg_inner=5,
        cg_eps=1e-5,
        gamma=0,
    ):
        if self.measurement_condition is None:
            raise ValueError("Measurement condition is not set.")
        if isinstance(self.unet.config.sample_size, int):
            image_shape = (
                batch_size,
                self.unet.config.in_channels,
                self.unet.config.sample_size,
                self.unet.config.sample_size,
            )
        else:
            image_shape = (
                batch_size,
                self.unet.config.in_channels,
                *self.unet.config.sample_size,
            )
        image = randn_tensor(
            image_shape,
            generator=generator,
            device=self.device,
            dtype=self.unet.dtype,
        )
        if image.ndim == 4 and measurement.ndim == 3:
            measurement = measurement.unsqueeze(0)
        self.scheduler.set_timesteps(num_inference_steps)
        bcg = self.measurement_condition.operator.transpose(measurement)
        initial_residuals = []
        final_residuals = []
        cg_iterations_per_step = []
        diverged = False

        timestep_progress = self.progress_bar(self.scheduler.timesteps)
        for timestep in timestep_progress:
            with torch.no_grad():
                model_output = self.unet(image, timestep).sample
            out = self.scheduler.step(
                model_output, timestep, image, generator=generator
            )
            x0_prediction = out.pred_original_sample
            if gamma > 0:
                bcg = (
                    x0_prediction
                    + gamma * self.measurement_condition.operator.transpose(measurement)
                )
            residual = (
                bcg
                - self.measurement_condition.operator.transpose(
                    self.measurement_condition.operator(x0_prediction)
                )
                - gamma * x0_prediction
            )
            direction = residual.clone()
            residual_squared = torch.sum(residual.reshape(-1) ** 2)
            initial_residuals.append(float(torch.sqrt(residual_squared).detach().cpu()))
            completed = 0

            for _ in range(int(cg_inner)):
                normal_direction = (
                    self.measurement_condition.operator.transpose(
                        self.measurement_condition.operator(direction)
                    )
                    + gamma * direction
                )
                denominator = torch.sum(
                    direction.reshape(-1) * normal_direction.reshape(-1)
                )
                if not torch.isfinite(denominator) or abs(float(denominator)) < 1e-20:
                    diverged = True
                    break
                step_size = residual_squared / denominator
                x0_prediction = x0_prediction + step_size * direction
                residual = residual - step_size * normal_direction
                new_residual_squared = torch.sum(residual.reshape(-1) ** 2)
                completed += 1
                if not torch.isfinite(new_residual_squared):
                    diverged = True
                    break
                if float(torch.sqrt(new_residual_squared)) < float(cg_eps):
                    residual_squared = new_residual_squared
                    break
                direction = residual + (
                    new_residual_squared / residual_squared
                ) * direction
                residual_squared = new_residual_squared

            final_residual = float(torch.sqrt(residual_squared).detach().cpu())
            final_residuals.append(final_residual)
            cg_iterations_per_step.append(completed)
            if hasattr(timestep_progress, "set_postfix"):
                timestep_progress.set_postfix(
                    cg_total=int(sum(cg_iterations_per_step)),
                    residual=f"{final_residual:.3g}",
                )
            if diverged:
                raise FloatingPointError("DDS conjugate gradient diverged or broke down")

            previous_timestep = (
                timestep
                - self.scheduler.config.num_train_timesteps
                // self.scheduler.num_inference_steps
            )
            alpha_t = self.scheduler.alphas_cumprod[timestep]
            alpha_previous = (
                self.scheduler.alphas_cumprod[previous_timestep]
                if previous_timestep >= 0
                else self.scheduler.final_alpha_cumprod
            )
            variance = self.scheduler._get_variance(
                timestep, previous_timestep
            )
            standard_deviation = eta * variance ** 0.5
            sample_direction = (
                1 - alpha_previous - standard_deviation**2
            ) ** 0.5 * model_output
            previous_sample = (
                alpha_previous**0.5 * x0_prediction + sample_direction
            )
            if eta > 0:
                variance_noise = randn_tensor(
                    model_output.shape,
                    generator=generator,
                    device=model_output.device,
                    dtype=model_output.dtype,
                )
                previous_sample = previous_sample + standard_deviation * variance_noise
            image = previous_sample

        self.dds_diagnostics = {
            "initial_residuals": initial_residuals,
            "final_residuals": final_residuals,
            "initial_residual": initial_residuals[0] if initial_residuals else np.nan,
            "final_residual": final_residuals[-1] if final_residuals else np.nan,
            "cg_iterations_per_step": cg_iterations_per_step,
            "completed_cg_iterations": int(sum(cg_iterations_per_step)),
            "diverged": diverged,
        }
        image = image.detach().cpu().numpy()
        if not return_dict:
            return (image,)
        return ImagePipelineOutput(images=image)

def run_dds(ground_truth, measurement_bundle, operator_bundle, diffusion_model, method_config, seed):
    operator = CountingOperator(operator_bundle)
    scheduler = DDIMScheduler.from_config(BASE_SCHEDULER_CONFIG)
    pipeline = DiagnosticDDSPipeline(unet=diffusion_model, scheduler=scheduler).to(DEVICE)
    pipeline.set_progress_bar_config(
        disable=CONFIG.RUN_MODE != "validation",
        desc="dds: diffusion + CG",
        unit="step",
        position=1,
        leave=False,
        dynamic_ncols=True,
    )
    pipeline.measurement_condition = DDS(operator=operator, noiser=NoNoise())
    measurement = bundle_tensor(measurement_bundle, "log_line_integrals")

    def work():
        output = pipeline(
            num_inference_steps=method_config["num_inference_steps"],
            measurement=measurement,
            cg_inner=method_config["cg_inner"],
            cg_eps=method_config["cg_eps"],
            gamma=method_config["gamma"],
            eta=method_config["eta"],
            generator=make_generator(seed),
            output_type="np",
        )
        reconstruction = output_to_tensor(output)
        residual = relative_line_integral_residual(operator, reconstruction, measurement)
        return (
            reconstruction[0, 0].detach().cpu().numpy(),
            residual,
            dict(pipeline.dds_diagnostics),
        )

    payload, runtime, nfe, forward_calls, adjoint_calls = timed_work("dds", operator, work)
    del pipeline
    diagnostics = payload[2]
    diagnostics["final_relative_line_integral_residual"] = payload[1]
    return MethodResult(
        reconstruction=payload[0],
        runtime_seconds=runtime,
        nfe=nfe,
        method_hyperparameters={**method_config, "initialization_seed": seed},
        measurement_model="log_linearized_poisson",
        num_forward_operator_calls=forward_calls,
        num_adjoint_operator_calls=adjoint_calls,
        convergence_diagnostics=diagnostics,
        failure_status="success",
        error_message="",
    )


In [ ]:
def run_method(
    method_name,
    ground_truth,
    measurement_bundle,
    operator_bundle,
    diffusion_model,
    method_config,
    seed,
):
    started = time.perf_counter()
    try:
        set_all_seeds(seed)
        if method_name == "dps":
            result = run_dps(
                ground_truth, measurement_bundle, operator_bundle,
                diffusion_model, method_config, seed
            )
        elif method_name == "red_diff":
            result = run_red_diff(
                ground_truth, measurement_bundle, operator_bundle,
                diffusion_model, method_config, seed
            )
        elif method_name == "dmplug_linearized":
            result = run_dmplug(
                ground_truth, measurement_bundle, operator_bundle,
                diffusion_model, method_config, seed, poisson=False
            )
        elif method_name == "dmplug_poisson":
            result = run_dmplug(
                ground_truth, measurement_bundle, operator_bundle,
                diffusion_model, method_config, seed, poisson=True
            )
        elif method_name == "dds":
            result = run_dds(
                ground_truth, measurement_bundle, operator_bundle,
                diffusion_model, method_config, seed
            )
        else:
            raise KeyError(f"Unknown method: {method_name}")
        if result.reconstruction is None or not np.isfinite(result.reconstruction).all():
            raise FloatingPointError("Method did not return a finite reconstruction")
        return result
    except torch.cuda.OutOfMemoryError:
        message = traceback.format_exc()
        LOGGER.error("CUDA OOM in %s\n%s", method_name, message)
        gc.collect()
        torch.cuda.empty_cache()
        return MethodResult(
            reconstruction=None,
            runtime_seconds=time.perf_counter() - started,
            nfe=NFE_COUNTER.count,
            method_hyperparameters={**method_config, "initialization_seed": seed},
            measurement_model=(
                "raw_poisson_counts_beer_lambert"
                if method_name == "dmplug_poisson"
                else "log_linearized_poisson"
            ),
            num_forward_operator_calls=0,
            num_adjoint_operator_calls=0,
            convergence_diagnostics={},
            failure_status="oom",
            error_message=message,
        )
    except Exception:
        message = traceback.format_exc()
        LOGGER.error("Failure in %s\n%s", method_name, message)
        gc.collect()
        torch.cuda.empty_cache()
        return MethodResult(
            reconstruction=None,
            runtime_seconds=time.perf_counter() - started,
            nfe=NFE_COUNTER.count,
            method_hyperparameters={**method_config, "initialization_seed": seed},
            measurement_model=(
                "raw_poisson_counts_beer_lambert"
                if method_name == "dmplug_poisson"
                else "log_linearized_poisson"
            ),
            num_forward_operator_calls=0,
            num_adjoint_operator_calls=0,
            convergence_diagnostics={},
            failure_status="failed",
            error_message=message,
        )


## 15. Smoke test

Smoke mode performs a real one-slice reconstruction with every requested method, asserts finite final metrics, saves every output, and writes `smoke_test_output.json`. It is deliberately short and is not a scientific result.


In [ ]:
METRICS_PATH = OUTPUT_PATH / "metrics_per_image.csv"
REQUIRED_METRIC_COLUMNS = [
    "image_id", "method", "seed", "measurement_model", "checkpoint",
    "num_inference_steps", "nfe", "runtime_seconds", "forward_operator_calls",
    "adjoint_operator_calls", "cg_iterations", "optimizer_iterations",
    "psnr", "ssim", "lpips", "final_measurement_residual",
    "status", "error_message",
]
EXTRA_METRIC_COLUMNS = ["config_hash", "reconstruction_path", "diagnostics_path"]
ALL_METRIC_COLUMNS = REQUIRED_METRIC_COLUMNS + EXTRA_METRIC_COLUMNS

def expanded_method_names():
    names = []
    for method in CONFIG.METHODS:
        if method == "dmplug":
            if CONFIG.ENABLE_LINEARIZED_TRACK:
                names.append("dmplug_linearized")
            if CONFIG.ENABLE_POISSON_DMPLUG:
                names.append("dmplug_poisson")
        else:
            names.append(method)
    return names

def sanitized_method(method):
    return method.replace("/", "_").replace(" ", "_")

def reconstruction_paths(image_id, method):
    root = OUTPUT_PATH / "reconstructions" / sanitized_method(method)
    return root / f"{image_id}.npz", root / f"{image_id}.tif", root / f"{image_id}.json"

def valid_completed_result(image_id, method, frame):
    if frame.empty:
        return False
    match = frame[
        (frame["image_id"] == image_id)
        & (frame["method"] == method)
        & (frame["config_hash"] == CONFIG_HASH)
        & (frame["status"] == "success")
    ]
    if match.empty:
        return False
    npz_path, _, _ = reconstruction_paths(image_id, method)
    if not npz_path.exists():
        return False
    try:
        with np.load(npz_path, allow_pickle=False) as saved:
            return (
                str(saved["config_hash"].item()) == CONFIG_HASH
                and str(saved["method"].item()) == method
                and np.isfinite(saved["reconstruction"]).all()
            )
    except Exception:
        return False

def upsert_metric_row(row):
    if METRICS_PATH.exists():
        frame = pd.read_csv(METRICS_PATH, keep_default_na=False)
    else:
        frame = pd.DataFrame(columns=ALL_METRIC_COLUMNS)
    if not frame.empty:
        same = (
            (frame["image_id"] == row["image_id"])
            & (frame["method"] == row["method"])
            & (frame["config_hash"] == row["config_hash"])
        )
        frame = frame.loc[~same].copy()
    frame = pd.concat([frame, pd.DataFrame([row])], ignore_index=True)
    atomic_write_csv(frame[ALL_METRIC_COLUMNS], METRICS_PATH)

def save_reconstruction_and_diagnostics(image_id, method, result):
    npz_path, tif_path, diagnostics_path = reconstruction_paths(image_id, method)
    npz_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_npz = npz_path.with_suffix(".tmp.npz")
    np.savez_compressed(
        temporary_npz,
        reconstruction=np.asarray(result.reconstruction, dtype=np.float32),
        config_hash=np.array(CONFIG_HASH),
        method=np.array(method),
    )
    os.replace(temporary_npz, npz_path)
    if CONFIG.SAVE_RECONSTRUCTIONS:
        temporary_tif = tif_path.with_suffix(".tmp.tif")
        imwrite(temporary_tif, np.asarray(result.reconstruction, dtype=np.float32))
        os.replace(temporary_tif, tif_path)
    atomic_write_json(
        diagnostics_path,
        {
            "method_hyperparameters": result.method_hyperparameters,
            "measurement_model": result.measurement_model,
            "convergence_diagnostics": result.convergence_diagnostics,
            "runtime_seconds": result.runtime_seconds,
            "nfe": result.nfe,
        },
    )
    return npz_path, diagnostics_path

def result_row(image_id, method, seed, result, ground_truth):
    if result.failure_status == "success":
        metrics = compute_final_metrics(
            ground_truth, result.reconstruction, CONFIG.COMPUTE_LPIPS
        )
        npz_path, diagnostics_path = save_reconstruction_and_diagnostics(
            image_id, method, result
        )
        diagnostics = result.convergence_diagnostics
        final_residual = diagnostics.get(
            "final_measurement_residual",
            diagnostics.get("final_relative_line_integral_residual", np.nan),
        )
    else:
        metrics = {"psnr": np.nan, "ssim": np.nan, "lpips": np.nan}
        npz_path, diagnostics_path = "", ""
        final_residual = np.nan
    hyperparameters = result.method_hyperparameters
    return {
        "image_id": image_id,
        "method": method,
        "seed": seed,
        "measurement_model": result.measurement_model,
        "checkpoint": f"{CONFIG.MODEL_PATH_OR_HF_ID}@{CHECKPOINT_REVISION}",
        "num_inference_steps": hyperparameters.get("num_inference_steps", np.nan),
        "nfe": result.nfe,
        "runtime_seconds": result.runtime_seconds,
        "forward_operator_calls": result.num_forward_operator_calls,
        "adjoint_operator_calls": result.num_adjoint_operator_calls,
        "cg_iterations": result.convergence_diagnostics.get(
            "completed_cg_iterations", 0
        ),
        "optimizer_iterations": result.convergence_diagnostics.get(
            "completed_optimizer_iterations", 0
        ),
        "psnr": metrics["psnr"],
        "ssim": metrics["ssim"],
        "lpips": metrics["lpips"],
        "final_measurement_residual": final_residual,
        "status": result.failure_status,
        "error_message": result.error_message,
        "config_hash": CONFIG_HASH,
        "reconstruction_path": str(npz_path),
        "diagnostics_path": str(diagnostics_path),
    }

def execute_primary_experiment(records):
    current = (
        pd.read_csv(METRICS_PATH, keep_default_na=False)
        if METRICS_PATH.exists()
        else pd.DataFrame(columns=ALL_METRIC_COLUMNS)
    )
    methods = expanded_method_names()
    for image_index, record in enumerate(records):
        image_id = record["image_id"]
        bundle = MEASUREMENT_BUNDLES[image_id]
        ground_truth = bundle["ground_truth"].numpy()
        method_seed = CONFIG.BASE_SEED + image_index
        for method in methods:
            if CONFIG.RESUME and valid_completed_result(image_id, method, current):
                LOGGER.info("Resume: valid completed result %s / %s", image_id, method)
                continue
            base_name = "dmplug" if method.startswith("dmplug_") else method
            method_config = dict(CONFIG.METHOD_CONFIGS[base_name])
            LOGGER.info("Running %s / %s", image_id, method)
            result = run_method(
                method,
                ground_truth,
                bundle,
                BASE_OPERATOR,
                DIFFUSION_UNET,
                method_config,
                method_seed,
            )
            row = result_row(image_id, method, method_seed, result, ground_truth)
            upsert_metric_row(row)
            current = pd.read_csv(METRICS_PATH, keep_default_na=False)
            LOGGER.info(
                "Finished %s / %s: status=%s nfe=%s runtime=%.3fs psnr=%s ssim=%s",
                image_id, method, row["status"], row["nfe"], row["runtime_seconds"],
                row["psnr"], row["ssim"],
            )
            gc.collect()
            torch.cuda.empty_cache()
    return pd.read_csv(METRICS_PATH, keep_default_na=False)

if CONFIG.RUN_MODE == "smoke":
    SMOKE_RESULTS = execute_primary_experiment(IMAGE_RECORDS)
    expected = set(expanded_method_names())
    smoke_rows = SMOKE_RESULTS[
        (SMOKE_RESULTS["config_hash"] == CONFIG_HASH)
        & (SMOKE_RESULTS["image_id"].isin(CONFIG.IMAGE_IDS))
        & (SMOKE_RESULTS["method"].isin(expected))
    ].copy()
    failures = smoke_rows[smoke_rows["status"] != "success"]
    if set(smoke_rows["method"]) != expected or not failures.empty:
        raise RuntimeError(
            f"Smoke test did not complete every method. Failures:\n{failures.to_string(index=False)}"
        )
    for metric in ["psnr", "ssim", "runtime_seconds", "nfe"]:
        if not np.isfinite(pd.to_numeric(smoke_rows[metric])).all():
            raise RuntimeError(f"Smoke test produced nonfinite {metric}")
    smoke_payload = {
        "status": "passed",
        "config_hash": CONFIG_HASH,
        "image_ids": CONFIG.IMAGE_IDS,
        "methods": sorted(expected),
        "rows": smoke_rows[REQUIRED_METRIC_COLUMNS].to_dict(orient="records"),
    }
    atomic_write_json(OUTPUT_PATH / "smoke_test_output.json", smoke_payload)
    print(json.dumps(smoke_payload, indent=2, default=str))
else:
    print(f"Smoke cell skipped because RUN_MODE={CONFIG.RUN_MODE!r}")


## 16. Fixed DDS validation

This cell runs only in `validation` mode. It performs no hyperparameter search and does not execute DMPlug. It evaluates DDS exactly once on the hash-verified `ct_slice_009.tif`.

**Exact source of the DDS settings**

| Setting used here | Value | Pinned source |
|---|---:|---|
| Diffusion steps | `100` | DM4CT's Low Dose Challenge DDS call sets `num_inference_steps=100` in [`README.md`, line 88](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/README.md#L88) |
| Inner CG limit | `5` | The same call sets `cg_inner=5` at [`README.md`, line 88](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/README.md#L88) |
| CG stopping tolerance | `1e-5` | The same call sets `cg_eps=1e-5` at [`README.md`, line 88](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/README.md#L88) |
| Data-consistency weight | `gamma=1` | The same call explicitly sets `gamma=1` at [`README.md`, line 88](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/README.md#L88). This intentionally overrides the class default `gamma=0` |
| Scheduler | `DDIMScheduler.from_config(checkpoint)` | DM4CT constructs DDS with DDIM in [`pipelines.py`, lines 1726-1735](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/pipelines.py#L1726-L1735) |
| DDIM stochasticity | `eta=0.85` | The pinned DDS pipeline default is stated in [`pipelines.py`, lines 1739-1750](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/pipelines.py#L1739-L1750) |

**NFE provenance.** NFE is not copied from DM4CT or a paper. The notebook measures it with a PyTorch forward pre-hook on the shared UNet and resets the counter immediately before the real DDS call. The pinned DDS loop performs one `self.unet(image, t)` evaluation per scheduler timestep in [`pipelines.py`, lines 1818-1824](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/pipelines.py#L1818-L1824), so 100 steps should normally produce NFE 100; the saved counter is authoritative. CG work is reported separately because the loop performs up to five CG iterations per diffusion step and may stop early at `cg_eps` as shown in [`pipelines.py`, lines 1826-1845](https://github.com/DM4CT/DM4CT/blob/49b3e5907178b56338d55c17944de17168a7a0a1/pipelines.py#L1826-L1845).

The live bar reports the single reconstruction's diffusion steps, cumulative completed CG iterations, and current CG residual. Final-iterate PSNR, SSIM, runtime, measured NFE, and completed CG counts are saved.


In [ ]:
FIXED_VALIDATION_METHODS = [
    "dds",
]

def run_validation_protocol():
    rows = []
    total_runs = len(FIXED_VALIDATION_METHODS) * len(IMAGE_RECORDS)
    overall_progress = tqdm(
        total=total_runs,
        desc="Fixed validation",
        unit="reconstruction",
        position=0,
        leave=True,
        dynamic_ncols=True,
    )
    try:
        for method in FIXED_VALIDATION_METHODS:
            base_name = "dmplug" if method.startswith("dmplug_") else method
            method_config = dict(CONFIG.METHOD_CONFIGS[base_name])
            image_scores = []
            total_runtime = 0.0
            total_nfe = 0
            total_cg_iterations = 0
            total_optimizer_iterations = 0

            for image_index, record in enumerate(IMAGE_RECORDS):
                image_id = record["image_id"]
                bundle = MEASUREMENT_BUNDLES[image_id]
                seed = CONFIG.BASE_SEED + image_index
                overall_progress.set_postfix(method=method, image=image_id)
                tqdm.write(
                    f"Starting {method} on {image_id} with fixed DM4CT settings"
                )
                result = run_method(
                    method,
                    bundle["ground_truth"].numpy(),
                    bundle,
                    BASE_OPERATOR,
                    DIFFUSION_UNET,
                    method_config,
                    seed,
                )
                score = {
                    "image_id": image_id,
                    "seed": seed,
                    "status": result.failure_status,
                    "runtime_seconds": result.runtime_seconds,
                    "nfe": result.nfe,
                    "cg_iterations": result.convergence_diagnostics.get(
                        "completed_cg_iterations", 0
                    ),
                    "optimizer_iterations": result.convergence_diagnostics.get(
                        "completed_optimizer_iterations", 0
                    ),
                    "error_message": result.error_message,
                }
                if result.failure_status == "success":
                    metrics = compute_final_metrics(
                        bundle["ground_truth"].numpy(), result.reconstruction, False
                    )
                    score.update({"psnr": metrics["psnr"], "ssim": metrics["ssim"]})
                else:
                    score.update({"psnr": np.nan, "ssim": np.nan})
                image_scores.append(score)
                total_runtime += result.runtime_seconds
                total_nfe += result.nfe
                total_cg_iterations += score["cg_iterations"]
                total_optimizer_iterations += score["optimizer_iterations"]
                overall_progress.update(1)
                overall_progress.set_postfix(
                    method=method,
                    image=image_id,
                    status=result.failure_status,
                )
                tqdm.write(
                    f"Finished {method} on {image_id}: "
                    f"status={result.failure_status}, "
                    f"PSNR={score['psnr']:.3f}, SSIM={score['ssim']:.4f}, "
                    f"runtime={result.runtime_seconds:.1f}s, NFE={result.nfe}"
                )

            successful_scores = [
                score for score in image_scores if score["status"] == "success"
            ]
            rows.append(
                {
                    "method": method,
                    "configuration_source": ("DM4CT README.md@49b3e590 line 88 and ""pipelines.py@49b3e590 lines 1726-1750; no search"),
                    "configuration": json.dumps(method_config, sort_keys=True),
                    "mean_validation_psnr": (
                        np.mean([score["psnr"] for score in successful_scores])
                        if successful_scores else np.nan
                    ),
                    "mean_validation_ssim": (
                        np.mean([score["ssim"] for score in successful_scores])
                        if successful_scores else np.nan
                    ),
                    "successful_slices": len(successful_scores),
                    "failed_slices": len(image_scores) - len(successful_scores),
                    "total_runtime_seconds": total_runtime,
                    "total_nfe": total_nfe,
                    "total_cg_iterations": total_cg_iterations,
                    "total_optimizer_iterations": total_optimizer_iterations,
                    "per_image_scores": json.dumps(image_scores),
                    "selection_criterion": "none; fixed configuration",
                    "nfe_definition": (
                        "Measured UNet forward-pre-hook count during the real DDS call; "
                        "not inferred from the configured step count"
                    ),
                    "selected": True,
                }
            )
            atomic_write_csv(pd.DataFrame(rows), OUTPUT_PATH / "validation_results.csv")
    finally:
        overall_progress.close()

    frame = pd.DataFrame(rows)
    fixed_configurations = {
        row["method"]: json.loads(row["configuration"]) for row in rows
    }
    atomic_write_json(
        OUTPUT_PATH / "selected_validation_hyperparameters.json",
        {
            "selected": fixed_configurations,
            "selection_performed": False,
            "configuration_source": ("https://github.com/DM4CT/DM4CT/blob/""49b3e5907178b56338d55c17944de17168a7a0a1/README.md#L88"),
            "note": (
                "No hyperparameter sweep or metric-based selection was performed. "
                "Full-mode settings remain frozen and are not edited automatically."
            ),
        },
    )
    return frame

if CONFIG.RUN_MODE == "validation":
    VALIDATION_RESULTS = run_validation_protocol()
    display(VALIDATION_RESULTS)
else:
    empty_validation = pd.DataFrame(
        columns=[
            "method", "configuration_source", "configuration",
            "mean_validation_psnr", "mean_validation_ssim",
            "successful_slices", "failed_slices", "total_runtime_seconds",
            "total_nfe", "total_cg_iterations", "total_optimizer_iterations",
            "per_image_scores", "selection_criterion", "nfe_definition", "selected",
        ]
    )
    if not (OUTPUT_PATH / "validation_results.csv").exists():
        atomic_write_csv(empty_validation, OUTPUT_PATH / "validation_results.csv")
    print("Validation skipped; no test metrics were used for parameter selection.")


## 17. Full resumable experiment

Full mode processes the fixed 100 L506 IDs with frozen settings. A valid config-hashed reconstruction and success row are required for a resume skip. Rows and reconstructions are written atomically after each image/method, so a disconnect loses at most the active reconstruction.


In [ ]:
if CONFIG.RUN_MODE == "full":
    FULL_RESULTS = execute_primary_experiment(IMAGE_RECORDS)
    display(
        FULL_RESULTS[
            (FULL_RESULTS["config_hash"] == CONFIG_HASH)
            & (FULL_RESULTS["image_id"].isin(CONFIG.IMAGE_IDS))
        ]
    )
else:
    print(f"Full run skipped because RUN_MODE={CONFIG.RUN_MODE!r}")


### Preserved supplied DiffPDHG launcher (optional)

The supplied notebook's DiffPDHG execution remains available behind `RUN_REFERENCE_DIFFPDHG=False`. It is not reimplemented and is not mixed automatically into the DM4CT tables because it uses a custom non-ASTRA raw-count operator, a different attenuation mapping, `I0=10000`, and a different measurement cache.


In [ ]:
if RUN_REFERENCE_DIFFPDHG:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(DIFFPDHG_DIR / "requirements-colab-ct.txt")],
        check=True,
    )
    reference_output = OUTPUT_PATH / "reference_diffpdhg_rawcount"
    reference_command = [
        sys.executable,
        str(DIFFPDHG_DIR / "recover_inverse2.py"),
        "--config-name",
        "default_ct.yaml",
        f"hydra.run.dir={(reference_output / 'hydra').as_posix()}",
        "seed=99",
        "total_images=1",
        "batch_size=1",
        f"save_dir={reference_output.as_posix()}",
        "sampler.annealing_scheduler_config.num_steps=400",
        "inverse_task.admm_config.max_iter=400",
        "inverse_task.admm_config.pdhg.tau=0.01",
        "inverse_task.admm_config.pdhg.sigma_dual=1200.0",
        "inverse_task.operator.I0=10000.0",
        "inverse_task.operator.num_angles=80",
        "inverse_task.operator.num_detectors=512",
        "inverse_task.operator.measurement_mode=poisson",
        "model.model_config.model_id=jiayangshi/lodochallenge_pixel_diffusion",
        f"data.image_root_path={(DIFFPDHG_DIR / 'demo-samples' / 'ct_l067_subset_tiff').as_posix()}",
        "data.start_idx=1",
        "data.end_idx=2",
    ]
    subprocess.run(reference_command, cwd=DIFFPDHG_DIR, check=True)
else:
    print("Supplied DiffPDHG raw-count reference launcher retained but disabled.")


## 18. Aggregate metrics and confidence intervals

Summary statistics use successful final reconstructions. Bootstrap resampling uses the common successful image IDs across methods, preserving pairing. Optional DiffPDHG differences are emitted only when an external matching-ID CSV is configured, and compatibility is labelled explicitly.


In [ ]:
SUMMARY_PATH = OUTPUT_PATH / "metrics_summary.csv"
PAIRWISE_PATH = OUTPUT_PATH / "paired_differences_vs_diffpdhg.csv"

def bootstrap_mean_ci(values, seed=99, draws=2000):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return np.nan, np.nan
    generator = np.random.default_rng(seed)
    samples = generator.choice(values, size=(draws, values.size), replace=True).mean(axis=1)
    return tuple(np.quantile(samples, [0.025, 0.975]))

if METRICS_PATH.exists():
    metrics_frame = pd.read_csv(METRICS_PATH, keep_default_na=False)
    scoped = metrics_frame[
        (metrics_frame["config_hash"] == CONFIG_HASH)
        & (metrics_frame["image_id"].isin(CONFIG.IMAGE_IDS))
    ].copy()
    for column in [
        "psnr", "ssim", "lpips", "runtime_seconds", "nfe",
        "forward_operator_calls", "adjoint_operator_calls",
        "cg_iterations", "optimizer_iterations",
    ]:
        scoped[column] = pd.to_numeric(scoped[column], errors="coerce")
    successful = scoped[scoped["status"] == "success"].copy()
    method_id_sets = [
        set(group["image_id"]) for _, group in successful.groupby("method")
    ]
    common_ids = set.intersection(*method_id_sets) if method_id_sets else set()
    summary_rows = []
    for method, group in scoped.groupby("method"):
        success_group = group[
            (group["status"] == "success") & (group["image_id"].isin(common_ids))
        ]
        row = {
            "method": method,
            "measurement_model": (
                success_group["measurement_model"].iloc[0] if len(success_group) else ""
            ),
            "successful_reconstructions": int((group["status"] == "success").sum()),
            "failures": int((group["status"] != "success").sum()),
            "paired_successful_images": len(success_group),
        }
        for metric in [
            "psnr", "ssim", "lpips", "runtime_seconds", "nfe",
            "forward_operator_calls", "adjoint_operator_calls",
            "cg_iterations", "optimizer_iterations",
        ]:
            values = success_group[metric].dropna().to_numpy(dtype=np.float64)
            low, high = bootstrap_mean_ci(
                values, seed=CONFIG.BASE_SEED + sum(ord(c) for c in method + metric)
            )
            row.update(
                {
                    f"{metric}_mean": np.mean(values) if values.size else np.nan,
                    f"{metric}_std": np.std(values, ddof=1) if values.size > 1 else 0.0 if values.size else np.nan,
                    f"{metric}_median": np.median(values) if values.size else np.nan,
                    f"{metric}_iqr": (
                        np.quantile(values, 0.75) - np.quantile(values, 0.25)
                        if values.size else np.nan
                    ),
                    f"{metric}_paired_bootstrap_ci95_low": low,
                    f"{metric}_paired_bootstrap_ci95_high": high,
                }
            )
        summary_rows.append(row)
    summary_frame = pd.DataFrame(summary_rows)
else:
    scoped = pd.DataFrame(columns=ALL_METRIC_COLUMNS)
    summary_frame = pd.DataFrame()
atomic_write_csv(summary_frame, SUMMARY_PATH)
display(summary_frame)

pairwise_columns = [
    "image_id", "method", "metric", "diffusion_value", "diffpdhg_value",
    "difference_method_minus_diffpdhg", "measurement_compatible", "compatibility_note",
]
pairwise_rows = []
if DIFFPDHG_METRICS_CSV and Path(DIFFPDHG_METRICS_CSV).exists():
    diffpdhg = pd.read_csv(DIFFPDHG_METRICS_CSV)
    required = {"image_id", "psnr", "ssim"}
    if not required.issubset(diffpdhg.columns):
        raise ValueError(f"DiffPDHG CSV must contain {sorted(required)}")
    for _, row in scoped[scoped["status"] == "success"].iterrows():
        reference = diffpdhg[diffpdhg["image_id"] == row["image_id"]]
        if reference.empty:
            continue
        for metric in ["psnr", "ssim"]:
            pairwise_rows.append(
                {
                    "image_id": row["image_id"],
                    "method": row["method"],
                    "metric": metric,
                    "diffusion_value": float(row[metric]),
                    "diffpdhg_value": float(reference.iloc[0][metric]),
                    "difference_method_minus_diffpdhg": float(row[metric])
                    - float(reference.iloc[0][metric]),
                    "measurement_compatible": False,
                    "compatibility_note": (
                        "Supplied DiffPDHG is custom-projector raw Poisson; DM4CT primary paths are "
                        "ASTRA log-linearized. Difference is descriptive, not likelihood-matched."
                    ),
                }
            )
atomic_write_csv(pd.DataFrame(pairwise_rows, columns=pairwise_columns), PAIRWISE_PATH)


In [ ]:
strict_rows = []
extended_rows = []
if not summary_frame.empty:
    for _, row in summary_frame.iterrows():
        display_name = {
            "dps": "DPS",
            "red_diff": "RED-diff",
            "dmplug_linearized": "DMPlug (log-linearized)",
            "dmplug_poisson": "DMPlug (Poisson counts)",
            "dds": "DDS (log-linearized)",
        }.get(row["method"], row["method"])
        table_row = {
            "Method": display_name,
            "Diffusion checkpoint": f"{CONFIG.MODEL_PATH_OR_HF_ID}@{CHECKPOINT_REVISION[:8]}",
            "Measurement representation": row["measurement_model"],
            "Data-consistency model": {
                "dps": "linear ASTRA residual guidance",
                "red_diff": "linear ASTRA squared loss",
                "dmplug_linearized": "linear ASTRA MSE through generator",
                "dmplug_poisson": "Beer-Lambert Poisson NLL through generator",
                "dds": "linear ASTRA conjugate gradient",
            }.get(row["method"], ""),
            "NFE": row.get("nfe_mean", np.nan),
            "Runtime": row.get("runtime_seconds_mean", np.nan),
            "PSNR": row.get("psnr_mean", np.nan),
            "SSIM": row.get("ssim_mean", np.nan),
        }
        extended_rows.append(table_row)
        if row["measurement_model"] == "log_linearized_poisson":
            strict_rows.append(table_row)

strict_table = pd.DataFrame(strict_rows)
extended_table = pd.DataFrame(extended_rows)
atomic_write_csv(strict_table, OUTPUT_PATH / "strict_common_measurement_table.csv")
atomic_write_csv(extended_table, OUTPUT_PATH / "extended_diffusion_prior_table.csv")
print("Strict common-measurement table")
display(strict_table)
print("Extended diffusion-prior table")
display(extended_table)


## 19. Qualitative comparison

IDs are fixed above before results are inspected. All ground truth and reconstruction panels use the same `[-1, 1]` grayscale window. The sinogram column uses one per-slice sinogram scale because it is a different physical domain. Metrics under each reconstruction are final-iterate values.


In [ ]:
def load_saved_reconstruction(image_id, method):
    npz_path, _, _ = reconstruction_paths(image_id, method)
    if not npz_path.exists():
        return None
    with np.load(npz_path, allow_pickle=False) as saved:
        if str(saved["config_hash"].item()) != CONFIG_HASH:
            return None
        return np.asarray(saved["reconstruction"], dtype=np.float32)

def load_external_diffpdhg_reconstruction(image_id):
    if not DIFFPDHG_RECON_ROOT:
        return None
    root = Path(DIFFPDHG_RECON_ROOT)
    candidates = [
        root / f"{image_id}.npy",
        root / f"{image_id}.npz",
        root / f"{image_id}.tif",
        root / f"{image_id}.tiff",
    ]
    for path in candidates:
        if not path.exists():
            continue
        if path.suffix == ".npy":
            array = np.load(path, allow_pickle=False)
        elif path.suffix == ".npz":
            with np.load(path, allow_pickle=False) as saved:
                key = "reconstruction" if "reconstruction" in saved.files else saved.files[0]
                array = saved[key]
        else:
            array = imread(path)
        array = np.asarray(array, dtype=np.float32).squeeze()
        if array.shape != (512, 512) or not np.isfinite(array).all():
            raise ValueError(f"Invalid external DiffPDHG reconstruction: {path}")
        return array
    return None

available_qualitative_ids = [
    image_id
    for image_id in QUALITATIVE_IMAGE_IDS
    if image_id in MEASUREMENT_BUNDLES
]
if CONFIG.RUN_MODE == "smoke" and not available_qualitative_ids:
    available_qualitative_ids = CONFIG.IMAGE_IDS[:1]

panel_methods = expanded_method_names()
include_diffpdhg = any(
    load_external_diffpdhg_reconstruction(image_id) is not None
    for image_id in available_qualitative_ids
)
if available_qualitative_ids and METRICS_PATH.exists():
    panel_frame = pd.read_csv(METRICS_PATH, keep_default_na=False)
    columns = (
        ["Ground truth", "Log sinogram"]
        + panel_methods
        + (["DiffPDHG"] if include_diffpdhg else [])
    )
    figure, axes = plt.subplots(
        len(available_qualitative_ids),
        len(columns),
        figsize=(3.0 * len(columns), 3.4 * len(available_qualitative_ids)),
        squeeze=False,
    )
    for row_index, image_id in enumerate(available_qualitative_ids):
        bundle = MEASUREMENT_BUNDLES[image_id]
        ground_truth = bundle["ground_truth"].numpy().squeeze()
        axes[row_index, 0].imshow(ground_truth, cmap="gray", vmin=-1, vmax=1)
        axes[row_index, 0].set_title("Ground truth")
        axes[row_index, 0].set_xlabel(image_id)
        sinogram = bundle["log_line_integrals"].numpy().squeeze()
        axes[row_index, 1].imshow(sinogram, cmap="gray", aspect="auto")
        axes[row_index, 1].set_title("Log sinogram")
        axes[row_index, 1].set_xlabel(
            f"{CONFIG.NUM_ANGLES} views x {CONFIG.NUM_DETECTORS} bins"
        )
        for column_index, method in enumerate(panel_methods, start=2):
            reconstruction = load_saved_reconstruction(image_id, method)
            axes[row_index, column_index].set_title(
                {
                    "dps": "DPS",
                    "red_diff": "RED-diff",
                    "dmplug_linearized": "DMPlug",
                    "dmplug_poisson": "DMPlug Poisson",
                    "dds": "DDS",
                }.get(method, method)
            )
            if reconstruction is None:
                axes[row_index, column_index].text(
                    0.5, 0.5, "not available", ha="center", va="center"
                )
            else:
                axes[row_index, column_index].imshow(
                    reconstruction, cmap="gray", vmin=-1, vmax=1
                )
                metric_row = panel_frame[
                    (panel_frame["config_hash"] == CONFIG_HASH)
                    & (panel_frame["image_id"] == image_id)
                    & (panel_frame["method"] == method)
                    & (panel_frame["status"] == "success")
                ]
                if not metric_row.empty:
                    metric_row = metric_row.iloc[0]
                    axes[row_index, column_index].set_xlabel(
                        f"PSNR {float(metric_row['psnr']):.2f} | "
                        f"SSIM {float(metric_row['ssim']):.3f}\n"
                        f"{float(metric_row['runtime_seconds']):.1f}s | "
                        f"NFE {int(float(metric_row['nfe']))}"
                    )
        if include_diffpdhg:
            column_index = len(columns) - 1
            axes[row_index, column_index].set_title("DiffPDHG")
            reconstruction = load_external_diffpdhg_reconstruction(image_id)
            if reconstruction is None:
                axes[row_index, column_index].text(
                    0.5, 0.5, "not available", ha="center", va="center"
                )
            else:
                axes[row_index, column_index].imshow(
                    reconstruction, cmap="gray", vmin=-1, vmax=1
                )
                if DIFFPDHG_METRICS_CSV and Path(DIFFPDHG_METRICS_CSV).exists():
                    reference_metrics = pd.read_csv(DIFFPDHG_METRICS_CSV)
                    reference_row = reference_metrics[
                        reference_metrics["image_id"] == image_id
                    ]
                    if not reference_row.empty:
                        reference_row = reference_row.iloc[0]
                        runtime = pd.to_numeric(
                            pd.Series([reference_row.get("runtime_seconds", np.nan)]),
                            errors="coerce",
                        ).iloc[0]
                        nfe = pd.to_numeric(
                            pd.Series([reference_row.get("nfe", np.nan)]),
                            errors="coerce",
                        ).iloc[0]
                        axes[row_index, column_index].set_xlabel(
                            f"PSNR {float(reference_row['psnr']):.2f} | "
                            f"SSIM {float(reference_row['ssim']):.3f}\n"
                            f"{float(runtime):.1f}s | NFE {int(float(nfe))}"
                            if np.isfinite(float(runtime)) and np.isfinite(float(nfe))
                            else (
                                f"PSNR {float(reference_row['psnr']):.2f} | "
                                f"SSIM {float(reference_row['ssim']):.3f}"
                            )
                        )
        for axis in axes[row_index]:
            axis.set_xticks([])
            axis.set_yticks([])
    figure.tight_layout()
    panel_png = OUTPUT_PATH / "qualitative_comparison.png"
    panel_pdf = OUTPUT_PATH / "qualitative_comparison.pdf"
    figure.savefig(panel_png, dpi=180, bbox_inches="tight")
    figure.savefig(panel_pdf, bbox_inches="tight")
    plt.show()
    print(panel_png, panel_pdf)
else:
    print("No completed primary reconstructions are available for a qualitative panel.")


## 20. Export and run summary

The output root contains the experiment/config manifests, shared measurements, atomic per-image reconstructions, diagnostics, all requested CSVs, logs, smoke output, and qualitative panels. Copy or point `OUTPUT_ROOT` to Google Drive for persistent full runs.


In [ ]:
required_outputs = [
    "metrics_per_image.csv",
    "metrics_summary.csv",
    "paired_differences_vs_diffpdhg.csv",
    "experiment_config.json",
    "environment_manifest.json",
    "validation_results.csv",
    "run_log.txt",
]
output_status = {
    name: {
        "exists": (OUTPUT_PATH / name).exists(),
        "path": str(OUTPUT_PATH / name),
    }
    for name in required_outputs
}
run_summary = {
    "run_mode": CONFIG.RUN_MODE,
    "config_hash": CONFIG_HASH,
    "image_ids": CONFIG.IMAGE_IDS,
    "methods": expanded_method_names(),
    "outputs": output_status,
    "dds_same_measurement_model_as_supplied_diffpdhg": False,
    "dds_measurement_statement": (
        "DDS is evaluated on the shared DM4CT log-linearized ASTRA sinogram. "
        "The supplied DiffPDHG reference uses raw photon counts with a custom non-ASTRA operator."
    ),
}
atomic_write_json(OUTPUT_PATH / "run_summary.json", run_summary)
print(json.dumps(run_summary, indent=2))


## Changes relative to the reference notebook

- **Added imports:** ASTRA, DM4CT pipeline/conditioning classes, diffusers schedulers, tifffile, pandas, scikit-image metrics, optional LPIPS, plotting, hashing, dataclasses, logging, traceback, and reproducibility utilities.
- **Added repository files:** three fixed validation TIFFs are published under `validation_data/` at commit `d033e80...` and downloaded with SHA-256 verification only in validation mode. No DM4CT source files are copied into this notebook; it checks out official DM4CT at `49b3e59...` and retains the supplied DiffPDHG repository at `35db526...`. The only local DDS code is a diagnostic subclass of the official pinned pipeline loop.
- **Changed package versions:** ASTRA is changed from DM4CT's 2.3.0 environment to 2.5.0 for current Colab/PyPI CUDA-wheel support. DM4CT's diffusers 0.32.2 stack is retained; tqdm 4.67.1 is pinned for live validation progress. Colab supplies PyTorch/CUDA and their actual versions are manifested.
- **Added method adapters:** unified `run_method` plus `run_dps`, `run_red_diff`, `run_dmplug`, and `run_dds`; optional `dmplug_poisson`; actual NFE/operator counters; DDS CG diagnostics; DMPlug loss recording.
- **Added configuration entries:** all required run/data/model/geometry/noise/seed/output/resume/LPIPS/measurement-track fields, three run modes, fixed test and validation IDs, pinned validation-data repository/commit/checksums, frozen DDS settings, and line-level source provenance. The unused DDS `sirt_iterations` metadata field was removed because DDS does not call the pseudo-inverse path.
- **Added evaluation outputs:** cached measurement bundles, `metrics_per_image.csv`, aggregate/bootstrap summaries, optional DiffPDHG differences, strict and extended tables, a single fixed DDS validation record for `ct_slice_009` with live CG progress, manifests, run log, atomic reconstructions/diagnostics, and PNG/PDF qualitative panels.
- **Necessary DPS/RED-diff modifications:** none to their DM4CT algorithms. They are wrapped only for common inputs, exact counters, timing, final-residual diagnostics, and shortened smoke settings. The previously requested untimed UNet warm-up has been removed; every model call now belongs to a real reconstruction.
- **Reference DiffPDHG cell:** retained behind `RUN_REFERENCE_DIFFPDHG`; its original raw-count/non-ASTRA/`I0=10000` contract remains separate and disabled by default.
